# songo-model-stockfish-for-google-collab - All In One

Notebook unique pour:
- monter Google Drive
- cloner ou mettre a jour le code depuis GitHub
- installer les dependances
- lancer la generation de dataset
- construire le dataset final
- entrainer le modele
- evaluer le modele
- benchmarker le modele
- reprendre un job interrompu avec le meme `job_id`


## 1. Monter Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Configuration du projet

In [ ]:
GIT_REPO_URL = 'https://github.com/TON-COMPTE/songo-model-stockfish-for-google-collab.git'
GIT_BRANCH = 'main'
PROJECT_NAME = 'songo-model-stockfish-for-google-collab'
DRIVE_ROOT = '/content/drive/MyDrive/songo-stockfish'
WORKTREE = f'/content/{PROJECT_NAME}'

print('GIT_REPO_URL =', GIT_REPO_URL)
print('GIT_BRANCH   =', GIT_BRANCH)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('WORKTREE     =', WORKTREE)


## 3. Initialiser Drive

In [ ]:
import os

for path in [
    f'{DRIVE_ROOT}/code',
    f'{DRIVE_ROOT}/code_snapshots',
    f'{DRIVE_ROOT}/data/raw',
    f'{DRIVE_ROOT}/data/raw_colab_pro',
    f'{DRIVE_ROOT}/data/raw_full_matrix_colab_pro',
    f'{DRIVE_ROOT}/data/sampled',
    f'{DRIVE_ROOT}/data/sampled_colab_pro',
    f'{DRIVE_ROOT}/data/sampled_full_matrix_colab_pro',
    f'{DRIVE_ROOT}/data/datasets',
    f'{DRIVE_ROOT}/jobs',
    f'{DRIVE_ROOT}/models/checkpoints',
    f'{DRIVE_ROOT}/models/final',
    f'{DRIVE_ROOT}/models/promoted/best',
    f'{DRIVE_ROOT}/models/lineage',
    f'{DRIVE_ROOT}/logs',
    f'{DRIVE_ROOT}/reports/evaluations',
    f'{DRIVE_ROOT}/reports/benchmarks',
    f'{DRIVE_ROOT}/exports',
]:
    os.makedirs(path, exist_ok=True)

print('Drive layout ready')


## 4. Cloner ou mettre a jour le repo

In [ ]:
import os
import shutil
import subprocess

if not os.path.exists(os.path.join(WORKTREE, '.git')):
    if os.path.exists(WORKTREE):
        shutil.rmtree(WORKTREE)
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO_URL, WORKTREE], check=True)
else:
    subprocess.run(['git', '-C', WORKTREE, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', WORKTREE, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', WORKTREE, 'pull', '--ff-only', 'origin', GIT_BRANCH], check=True)

print('Worktree ready')


## 5. Installer les dependances

In [ ]:
import subprocess
subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-r', f'{WORKTREE}/requirements.txt'], check=True)
print('Environment ready in Colab runtime')


## 6. Choisir les configs et job ids

Le pipeline dataset supporte maintenant plusieurs modes de generation:
- `benchmatch`: generation longue a partir des matchs de reference
- `clone_existing`: creation d'une nouvelle source de dataset a partir d'un corpus deja existant
- `derive_existing`: creation d'une nouvelle variante filtree a partir d'un corpus existant
- `merge_existing`: fusion de plusieurs sources deja versionnees en une nouvelle grande source

Ensuite `dataset-build` peut choisir explicitement quelle source de dataset il enrichit.
Les originaux restent conserves, car chaque source et chaque dataset construit sont versionnes separement.
Tu peux aussi forcer un nouvel identifiant de dataset construit avec `DATASET_BUILD_ID_OVERRIDE`.


Pour comparer proprement deux familles de modeles, garde le meme dataset et change seulement `TRAIN_CONFIG` et les `JOB_ID`.

Exemple recommande:
- stable: `TRAIN_CONFIG = 'config/train.full_matrix.colab_pro.yaml'`
- residual: `TRAIN_CONFIG = 'config/train.full_matrix.colab_pro.residual.yaml'`

Important:
- utilise des `TRAIN_JOB_ID`, `EVALUATION_JOB_ID` et `BENCHMARK_JOB_ID` differents entre stable et residual
- ne reutilise pas le meme `TRAIN_JOB_ID` pour deux architectures differentes


In [ ]:
DATASET_GENERATE_CONFIG = 'config/dataset_generation.full_matrix.colab_pro.yaml'
DATASET_BUILD_CONFIG = 'config/dataset_build.full_matrix.colab_pro.yaml'
DATASET_MERGE_FINAL_CONFIG = 'config/dataset_merge_final.colab_pro.yaml'
TRAIN_CONFIG = 'config/train.full_matrix.colab_pro.yaml'
# Variante experimentale possible:
# TRAIN_CONFIG = 'config/train.full_matrix.colab_pro.residual.yaml'
EVALUATION_CONFIG = 'config/evaluation.full_matrix.colab_pro.yaml'
BENCHMARK_CONFIG = 'config/benchmark.colab_pro.yaml'

DATASET_GENERATE_JOB_ID = 'dataset_full_matrix_colab_pro_1m_001'
DATASET_BUILD_JOB_ID = 'dataset_build_full_matrix_colab_pro_1m_001'
DATASET_MERGE_FINAL_JOB_ID = 'dataset_merge_final_colab_pro_001'
TRAIN_JOB_ID = 'train_colab_pro_stable_001'
EVALUATION_JOB_ID = 'eval_colab_pro_stable_001'
BENCHMARK_JOB_ID = 'benchmark_colab_pro_stable_001'


In [ ]:
DATASET_GENERATION_MODE = 'benchmatch'
DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro'
SOURCE_DATASET_ID_FOR_GENERATION = ''
SOURCE_DATASET_IDS_FOR_GENERATION = []
DERIVATION_STRATEGY = 'unique_positions'
TARGET_SAMPLES = 1000000
SOURCE_DATASET_ID_FOR_BUILD = DATASET_SOURCE_ID
DATASET_BUILD_ID_OVERRIDE = ''
TARGET_LABELED_SAMPLES = 1000000
MERGED_FINAL_DATASET_ID = 'dataset_merged_final_colab_pro_insane_1m'
MERGE_ALL_BUILT_DATASETS = True
MERGE_SOURCE_DATASET_IDS = []

DATASET_GENERATE_EXTRA_ARGS = f"--generation-mode {DATASET_GENERATION_MODE} --dataset-source-id {DATASET_SOURCE_ID}"
if SOURCE_DATASET_ID_FOR_GENERATION:
    DATASET_GENERATE_EXTRA_ARGS += f" --source-dataset-id {SOURCE_DATASET_ID_FOR_GENERATION}"
if DATASET_GENERATION_MODE == 'merge_existing' and SOURCE_DATASET_IDS_FOR_GENERATION:
    joined_source_ids = ' '.join(SOURCE_DATASET_IDS_FOR_GENERATION)
    DATASET_GENERATE_EXTRA_ARGS += f" --source-dataset-ids {joined_source_ids}"
if DATASET_GENERATION_MODE == 'derive_existing':
    DATASET_GENERATE_EXTRA_ARGS += f" --derivation-strategy {DERIVATION_STRATEGY}"
if TARGET_SAMPLES:
    DATASET_GENERATE_EXTRA_ARGS += f" --target-samples {TARGET_SAMPLES}"

DATASET_BUILD_EXTRA_ARGS = ''
if SOURCE_DATASET_ID_FOR_BUILD:
    DATASET_BUILD_EXTRA_ARGS += f" --source-dataset-id {SOURCE_DATASET_ID_FOR_BUILD}"
if DATASET_BUILD_ID_OVERRIDE:
    DATASET_BUILD_EXTRA_ARGS += f" --dataset-id-override {DATASET_BUILD_ID_OVERRIDE}"
if TARGET_LABELED_SAMPLES:
    DATASET_BUILD_EXTRA_ARGS += f" --target-labeled-samples {TARGET_LABELED_SAMPLES}"

DATASET_MERGE_FINAL_EXTRA_ARGS = f"--dataset-id {MERGED_FINAL_DATASET_ID}"
if MERGE_ALL_BUILT_DATASETS:
    DATASET_MERGE_FINAL_EXTRA_ARGS += ' --include-all-built'
elif MERGE_SOURCE_DATASET_IDS:
    joined_ids = ' '.join(MERGE_SOURCE_DATASET_IDS)
    DATASET_MERGE_FINAL_EXTRA_ARGS += f" --source-dataset-ids {joined_ids}"

print('DATASET_GENERATION_MODE        =', DATASET_GENERATION_MODE)
print('DATASET_SOURCE_ID              =', DATASET_SOURCE_ID)
print('SOURCE_DATASET_ID_FOR_GENERATION =', SOURCE_DATASET_ID_FOR_GENERATION)
print('SOURCE_DATASET_IDS_FOR_GENERATION =', SOURCE_DATASET_IDS_FOR_GENERATION)
print('DERIVATION_STRATEGY            =', DERIVATION_STRATEGY)
print('TARGET_SAMPLES                =', TARGET_SAMPLES)
print('SOURCE_DATASET_ID_FOR_BUILD    =', SOURCE_DATASET_ID_FOR_BUILD)
print('DATASET_BUILD_ID_OVERRIDE      =', DATASET_BUILD_ID_OVERRIDE)
print('TARGET_LABELED_SAMPLES        =', TARGET_LABELED_SAMPLES)
print('MERGED_FINAL_DATASET_ID       =', MERGED_FINAL_DATASET_ID)
print('MERGE_ALL_BUILT_DATASETS      =', MERGE_ALL_BUILT_DATASETS)
print('MERGE_SOURCE_DATASET_IDS      =', MERGE_SOURCE_DATASET_IDS)
print('DATASET_GENERATE_EXTRA_ARGS    =', DATASET_GENERATE_EXTRA_ARGS)
print('DATASET_BUILD_EXTRA_ARGS       =', DATASET_BUILD_EXTRA_ARGS)
print('DATASET_MERGE_FINAL_EXTRA_ARGS =', DATASET_MERGE_FINAL_EXTRA_ARGS)


Presets utiles:

- Corpus long de reference par matchs:
  - `DATASET_GENERATION_MODE = 'benchmatch'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro'`
  - `SOURCE_DATASET_ID_FOR_GENERATION = ''`

- Nouvelle source rapide a partir du corpus 1M deja existant:
  - `DATASET_GENERATION_MODE = 'clone_existing'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro_variant_a'`
  - `SOURCE_DATASET_ID_FOR_GENERATION = 'sampled_full_matrix_colab_pro'`

- Variante derivee sans relancer les matchs:
  - `DATASET_GENERATION_MODE = 'derive_existing'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro_unique'`
  - `SOURCE_DATASET_ID_FOR_GENERATION = 'sampled_full_matrix_colab_pro'`
  - `DERIVATION_STRATEGY = 'unique_positions'`
  - `TARGET_SAMPLES = 2000000`

- Grande source fusionnee a partir de plusieurs sources deja versionnees:
  - `DATASET_GENERATION_MODE = 'merge_existing'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro_giant'`
  - `SOURCE_DATASET_IDS_FOR_GENERATION = ['sampled_full_matrix_colab_pro', 'sampled_full_matrix_colab_pro_unique']`
  - `TARGET_SAMPLES = 2000000`

- Build d'un nouveau dataset final a partir d'une source existante:
  - `SOURCE_DATASET_ID_FOR_BUILD = 'sampled_full_matrix_colab_pro_variant_a'`
  - `DATASET_BUILD_ID_OVERRIDE = 'dataset_v3_variant_a_insane_2m'`
  - `TARGET_LABELED_SAMPLES = 2000000`

- Fusionner tous les datasets finaux existants en un nouveau dataset final versionne:
  - `MERGED_FINAL_DATASET_ID = 'dataset_merged_final_colab_pro_insane_1m'`
  - `MERGE_ALL_BUILT_DATASETS = True`


Convention recommandee de nommage:
- sources: `sampled_full_matrix_colab_pro_<variante>`
- datasets finaux: `dataset_vN_full_matrix_colab_pro_<variante>_insane_1m`

Evite de reutiliser un identifiant pour un contenu different.


In [ ]:
from pathlib import Path
import json

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
if registry_path.exists():
    registry = json.loads(registry_path.read_text(encoding='utf-8'))
    print('Dataset sources:')
    for item in registry.get('dataset_sources', [])[-10:]:
        print('-', item.get('dataset_source_id'), '| mode=', item.get('source_mode'), '| parent=', item.get('source_dataset_id', ''), '| parents=', item.get('source_dataset_ids', []), '| samples=', item.get('sampled_positions'))
    print('\nBuilt datasets:')
    for item in registry.get('built_datasets', [])[-10:]:
        print('-', item.get('dataset_id'), '| build_mode=', item.get('build_mode', 'teacher_label'), '| source=', item.get('source_dataset_id'), '| teacher=', f"{item.get('teacher_engine')}:{item.get('teacher_level')}", '| labeled=', item.get('labeled_samples'))
else:
    print('Aucun dataset_registry.json trouve pour le moment')


Tu peux aussi utiliser directement la CLI `dataset-list` pour relire le registre avec le meme comportement que hors notebook.

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main dataset-list --config $DATASET_GENERATE_CONFIG --kind all"

## 7. Lancer la generation du dataset

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main dataset-generate --config $DATASET_GENERATE_CONFIG --job-id $DATASET_GENERATE_JOB_ID $DATASET_GENERATE_EXTRA_ARGS"

## 8. Construire le dataset final

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main dataset-build --config $DATASET_BUILD_CONFIG --job-id $DATASET_BUILD_JOB_ID $DATASET_BUILD_EXTRA_ARGS"

## 9. Fusionner des datasets finaux existants

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main dataset-merge-final --config $DATASET_MERGE_FINAL_CONFIG --job-id $DATASET_MERGE_FINAL_JOB_ID $DATASET_MERGE_FINAL_EXTRA_ARGS"

## 10. Entrainement GPU

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main train --config $TRAIN_CONFIG --job-id $TRAIN_JOB_ID"

## 11. Evaluation

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main evaluate --config $EVALUATION_CONFIG --job-id $EVALUATION_JOB_ID"

## 12. Benchmark

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main benchmark --config $BENCHMARK_CONFIG --job-id $BENCHMARK_JOB_ID"

## 11bis. Lire les resultats de comparaison

Ordre de lecture recommande:
- 1. `benchmark_score` et winrates benchmark
- 2. `policy_accuracy_top1`, `policy_accuracy_top3`, `value_mae`
- 3. `best_validation_metric`

Regle simple:
- si `residual` est meilleur en benchmark et pas moins bon en evaluation, il gagne
- s'il est meilleur en evaluation mais pas en benchmark, garde `stable`

Fichiers a comparer pour chaque variante:
- `jobs/<train_job_id>/training_summary.json`
- `jobs/<eval_job_id>/evaluation_summary.json`
- `jobs/<benchmark_job_id>/benchmark_summary.json`


## 12. Suivre un job en direct

In [ ]:
WATCH_JOB_ID = TRAIN_JOB_ID
!tail -f {DRIVE_ROOT}/jobs/{WATCH_JOB_ID}/events.jsonl

## 13. Reprendre un job interrompu

Relance simplement la meme cellule de commande avec le meme `job_id`.